# 🔧 Dependency Injection
### A Visual, Step-by-Step Guide

---

## What is Dependency Injection?

**Dependency Injection (DI)** means:
> *"Don't create what you need — ask someone to give it to you."*

### Real life analogy 🍕

```
❌ WITHOUT DI (bad):
   You (the function) go to the farm, grow wheat,
   mill the flour, make the dough, bake the pizza yourself.

✅ WITH DI (good):
   You just say "I need a pizza" and someone delivers it.
   You don't care HOW it was made.
```

In code terms:
- **Without DI**: your function creates its own database connection, config, etc.
- **With DI**: your function just says "I need a DB" and FastAPI gives it

---

## Why does it matter?

| Without DI | With DI |
|---|---|
| Hard to test (real DB always needed) | Easy to test (swap real DB with fake) |
| Code is tightly coupled | Code is loosely coupled |
| Change DB = change every file | Change DB = change one place |
| Duplicate code everywhere | One place, reused everywhere |

## Step 1 — Install libraries

In [ ]:
!pip install fastapi uvicorn matplotlib

## Step 2 — Without DI (the BAD way)

See how messy and hard to test this is:

In [ ]:
# ❌ WITHOUT Dependency Injection
# Every function creates its own dependencies

class DatabaseConnection:
    def __init__(self):
        # Imagine this connects to a real database
        print("  📡 Connecting to database...")
        self.connected = True

    def get_user(self, user_id: int):
        return {"id": user_id, "name": "Prakash", "email": "prakash@example.com"}


# ❌ Problem: each function creates its OWN database connection
def get_user_profile(user_id: int):
    db = DatabaseConnection()   # ← creates connection inside
    return db.get_user(user_id)

def get_user_documents(user_id: int):
    db = DatabaseConnection()   # ← creates ANOTHER connection
    user = db.get_user(user_id)
    return ["doc1.pdf", "doc2.pdf"]

def get_user_settings(user_id: int):
    db = DatabaseConnection()   # ← and ANOTHER one!
    return {"theme": "dark"}


print("❌ WITHOUT DI — 3 separate DB connections created:")
print("Calling get_user_profile:")
get_user_profile(1)
print("Calling get_user_documents:")
get_user_documents(1)
print("Calling get_user_settings:")
get_user_settings(1)

print("\n⚠️  Problems:")
print("  1. 3 DB connections for 1 request — wasteful")
print("  2. Can't swap DB for testing — it's hardcoded inside")
print("  3. If DB changes, must update every single function")

## Step 3 — With DI (the GOOD way)

In [ ]:
# ✅ WITH Dependency Injection
# Dependencies are passed IN from outside

class DatabaseConnection:
    def __init__(self, url: str = "postgresql://localhost/ragdb"):
        print(f"  📡 Connecting to: {url}")
        self.url = url
        self.connected = True

    def get_user(self, user_id: int):
        return {"id": user_id, "name": "Prakash", "email": "prakash@example.com"}


# ✅ Functions RECEIVE the DB — they don't create it
def get_user_profile(user_id: int, db: DatabaseConnection):
    return db.get_user(user_id)          # ← db is injected

def get_user_documents(user_id: int, db: DatabaseConnection):
    user = db.get_user(user_id)          # ← same db injected
    return ["doc1.pdf", "doc2.pdf"]

def get_user_settings(user_id: int, db: DatabaseConnection):
    return {"theme": "dark"}             # ← same db injected


print("✅ WITH DI — ONE DB connection shared:")
db = DatabaseConnection()   # ← created ONCE outside

profile   = get_user_profile(1, db)
documents = get_user_documents(1, db)
settings  = get_user_settings(1, db)

print(f"\nProfile:   {profile}")
print(f"Documents: {documents}")
print(f"Settings:  {settings}")

print("\n✅ Benefits:")
print("  1. ONE DB connection — efficient")
print("  2. Can swap DB for testing — just pass a fake one")
print("  3. Change DB = change one place only")

## Step 4 — DI for Testing (the real superpower)

In [ ]:
# The real power of DI — swap real dependencies with fakes in tests

class RealDatabase:
    """Real DB — connects to PostgreSQL"""
    def get_user(self, user_id: int):
        print("  📡 Querying real PostgreSQL...")
        return {"id": user_id, "name": "Prakash"}


class FakeDatabase:
    """Fake DB — for testing, no real connection needed"""
    def get_user(self, user_id: int):
        print("  🧪 Using fake test database...")
        return {"id": user_id, "name": "Test User"}


# Same function works with BOTH
def get_user_profile(user_id: int, db):
    return db.get_user(user_id)


print("🏭 PRODUCTION — uses real database:")
real_db = RealDatabase()
result = get_user_profile(1, db=real_db)
print(f"  Result: {result}")

print("\n🧪 TESTING — uses fake database (no PostgreSQL needed!):")
fake_db = FakeDatabase()
result = get_user_profile(1, db=fake_db)
print(f"  Result: {result}")

print("\n💡 The function get_user_profile() didn't change at all!")
print("   We just swapped what was injected into it.")

## Step 5 — FastAPI's `Depends()` — DI built into the framework

FastAPI has DI built in via `Depends()`. Here's how it works:

In [ ]:
from fastapi import FastAPI, Depends, HTTPException
from typing import Annotated

app = FastAPI()

# ── Dependency 1: Database session ────────────────────────────
def get_db():
    """FastAPI calls this and injects the result into your endpoint"""
    print("  📡 Opening DB session")
    db = {"connected": True, "users": {1: "Prakash", 2: "Ananya"}}
    try:
        yield db          # ← yield means it runs cleanup after request
    finally:
        print("  📡 Closing DB session")


# ── Dependency 2: Current user (from JWT token) ────────────────
def get_current_user(token: str = "fake-token"):
    """In real code this decodes the JWT token"""
    if token != "valid-token":
        raise HTTPException(status_code=401, detail="Invalid token")
    return {"id": 1, "email": "prakash@example.com"}


# ── Endpoint using BOTH dependencies ──────────────────────────
@app.get("/documents")
async def list_documents(
    db = Depends(get_db),                    # ← FastAPI injects DB
    current_user = Depends(get_current_user) # ← FastAPI injects user
):
    """
    FastAPI automatically:
    1. Calls get_db() and passes result as 'db'
    2. Calls get_current_user() and passes result as 'current_user'
    3. Cleans up after the request (closes DB session)
    """
    return {
        "user": current_user["email"],
        "documents": ["doc1.pdf", "doc2.pdf"]
    }


print("FastAPI endpoint defined with 2 dependencies:")
print("  1. get_db()           → provides database session")
print("  2. get_current_user() → provides authenticated user")
print()
print("When GET /documents is called, FastAPI automatically:")
print("  Step 1: calls get_db() → injects db")
print("  Step 2: calls get_current_user() → injects current_user")
print("  Step 3: runs your function with both injected")
print("  Step 4: closes DB session automatically")

## Step 6 — Chained Dependencies

Dependencies can depend on other dependencies:

In [ ]:
# Dependencies can depend on other dependencies!

def get_settings():
    """Level 1 — no dependencies"""
    print("  ⚙️  Loading settings")
    return {"db_url": "postgresql://localhost/ragdb", "secret": "abc123"}


def get_db(settings = Depends(get_settings)):
    """Level 2 — depends on settings"""
    print(f"  📡 Connecting to: {settings['db_url']}")
    return {"connected": True}


def get_current_user(settings = Depends(get_settings)):
    """Level 2 — also depends on settings"""
    print(f"  🔑 Verifying token with secret: {settings['secret']}")
    return {"id": 1, "email": "prakash@example.com"}


@app.get("/query")
async def query_documents(
    db = Depends(get_db),                    # depends on settings
    user = Depends(get_current_user),        # also depends on settings
):
    """Level 3 — depends on db AND current_user"""
    return {"answer": "RAG is cool!", "user": user["email"]}


print("Dependency chain for GET /query:")
print("""
GET /query
  ├── get_db()
  │     └── get_settings()   ← called first
  └── get_current_user()
        └── get_settings()   ← FastAPI is smart: called only ONCE,
                                result reused (cached per request)
""")
print("💡 FastAPI caches dependencies within a request.")
print("   get_settings() runs once even though 2 deps need it.")

## Step 7 — Visual Diagram

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.patch.set_facecolor("#f7f5f2")

colors = {
    "bad":      "#f8d7da",
    "good":     "#d4edda",
    "endpoint": "#2b5be0",
    "dep":      "#27ae60",
    "db":       "#e67e22",
    "arrow":    "#555",
}

def draw_box(ax, x, y, w, h, text, color, text_color="white", fontsize=10):
    rect = mpatches.FancyBboxPatch(
        (x - w/2, y - h/2), w, h,
        boxstyle="round,pad=0.1",
        facecolor=color, edgecolor="white",
        linewidth=2
    )
    ax.add_patch(rect)
    ax.text(x, y, text, ha="center", va="center",
            fontsize=fontsize, fontweight="bold", color=text_color)

def draw_arrow(ax, x1, y1, x2, y2, color="#555", label=""):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="->", color=color, lw=2))
    if label:
        mx, my = (x1+x2)/2, (y1+y2)/2
        ax.text(mx+0.1, my, label, fontsize=8, color=color)

# ── LEFT: Without DI ──────────────────────────────────────────
ax1 = axes[0]
ax1.set_xlim(0, 6)
ax1.set_ylim(0, 9)
ax1.axis("off")
ax1.set_facecolor("#f7f5f2")
ax1.set_title("❌ Without DI", fontsize=14,
              fontweight="bold", color="#c0392b", pad=12)

# Three endpoints
for i, (name, y) in enumerate([
    ("GET /profile", 7.5),
    ("GET /documents", 4.5),
    ("GET /settings", 1.5),
]):
    draw_box(ax1, 1.8, y, 2.8, 0.7, name, colors["endpoint"])
    draw_box(ax1, 4.5, y, 1.6, 0.6, "DB conn", colors["bad"], "#721c24", 9)
    draw_arrow(ax1, 3.2, y, 3.7, y, "#c0392b", "creates")

ax1.text(3, 0.3, "3 separate DB connections! Wasteful.",
         ha="center", fontsize=9, color="#c0392b", style="italic")

# ── RIGHT: With DI ────────────────────────────────────────────
ax2 = axes[1]
ax2.set_xlim(0, 6)
ax2.set_ylim(0, 9)
ax2.axis("off")
ax2.set_facecolor("#f7f5f2")
ax2.set_title("✅ With DI (FastAPI Depends)", fontsize=14,
              fontweight="bold", color="#27ae60", pad=12)

# Depends box
draw_box(ax2, 3, 7.5, 2.2, 0.7, "get_db()", colors["dep"])
ax2.text(3, 6.9, "(one shared session)",
         ha="center", fontsize=8, color="#27ae60", style="italic")

# Three endpoints
for name, y in [
    ("GET /profile", 5.5),
    ("GET /documents", 4.0),
    ("GET /settings", 2.5),
]:
    draw_box(ax2, 1.5, y, 2.6, 0.65, name, colors["endpoint"])
    draw_arrow(ax2, 3, 7.15, 1.5, y+0.33, colors["dep"], "inject")

# DB
draw_box(ax2, 4.8, 7.5, 1.8, 0.65, "PostgreSQL", colors["db"])
draw_arrow(ax2, 3, 7.5, 3.9, 7.5, colors["db"], "")

ax2.text(3, 0.3, "One DB session, shared efficiently!",
         ha="center", fontsize=9, color="#27ae60", style="italic")

plt.suptitle("Dependency Injection — Before vs After",
             fontsize=16, fontweight="bold", color="#1a1714", y=1.01)
plt.tight_layout()
plt.savefig("dependency_injection.png", dpi=150,
            bbox_inches="tight", facecolor="#f7f5f2")
plt.show()
print("✅ Diagram saved as dependency_injection.png")

## Step 8 — DI in your actual RAG project

In [ ]:
code = '''
# ── How DI is used in YOUR RAG project ────────────────────────

# 1. Database session dependency
async def get_db() -> AsyncGenerator:
    async with AsyncSessionLocal() as session:
        yield session                     # ← yields DB session
                                          #   closes after request

# 2. Current user dependency (uses JWT)
async def get_current_user(
    token: str = Depends(oauth2_scheme),  # ← depends on token
    db = Depends(get_db)                  # ← depends on DB
) -> User:
    payload = jwt.decode(token, SECRET_KEY)
    user = await db.get(User, payload["sub"])
    return user

# 3. Your endpoints just declare what they need
@router.post("/query")
async def query(
    request: QueryRequest,
    db: AsyncSession = Depends(get_db),              # ← DB injected
    current_user: User = Depends(get_current_user)   # ← User injected
):
    # FastAPI handles ALL the setup and teardown
    # You just use db and current_user directly
    results = await rag_pipeline(request.question, db)
    return {"answer": results, "user": current_user.email}


# Dependency chain for POST /query:
#
#   POST /query
#     ├── get_db()              → AsyncSession
#     └── get_current_user()
#           ├── oauth2_scheme   → JWT token from header
#           └── get_db()        → same session (cached!)
'''

print(code)

## Summary

| Concept | Meaning |
|---|---|
| **Dependency** | Something your function needs (DB, config, user) |
| **Injection** | Passing that thing in from outside |
| **`Depends()`** | FastAPI's way to declare and inject dependencies |
| **`yield`** | Allows cleanup after the request (close DB session) |
| **Caching** | FastAPI calls each dependency once per request |
| **Chaining** | Dependencies can depend on other dependencies |

---

### The 3 Golden Rules of DI

```
1. Don't create dependencies inside functions
   ❌  def my_func(): db = Database()
   ✅  def my_func(db = Depends(get_db)):

2. One dependency, many users
   ✅  get_db() → used by 20 endpoints, defined once

3. Swap for testing
   ✅  app.dependency_overrides[get_db] = get_fake_db
```

### In your RAG project, DI is used for:
- `get_db()` — PostgreSQL session
- `get_current_user()` — JWT auth + user lookup
- `get_settings()` — app configuration
- `get_qdrant_client()` — vector database client
- `get_redis_client()` — cache client